# **AI Hub 데이터 사용 실습 (Windows / macOS 로컬 PC)**

---

## **실습 환경 준비**

- (선택) Visual Studio Code를 열고,
- (터미널:Ctrl+j) 가상환경을 만들고
- (확장:Ctrl+Shift+X) Jupyter를 설치한 뒤 이 노트북을 연다.
- 아래 셀은 현재 노트북 커널에 실습 패키지를 설치한다.
- 가상환경을 사용하면 실습 패키지가 다른 프로젝트와 섞이지 않는다.
- 아래 `%pip` 셀은 **현재 노트북 커널**에 필요한 패키지를 설치한다.

| 운영체제 | 터미널에서 최초 1회 실행 |
|---|---|
| Windows PowerShell | `py -m venv .venv` → `.\.venv\Scripts\Activate.ps1` → `python -m pip install jupyter` → `jupyter notebook` |
| macOS 터미널 | `python3 -m venv .venv` → `source .venv/bin/activate` → `python -m pip install jupyter` → `jupyter notebook` |

> VS Code를 사용한다면 `Python`, `Jupyter` 확장을 설치한 뒤, 우측 상단에서 `.venv` 커널을 선택한다.

In [ ]:
%pip install -q -U numpy pandas matplotlib scikit-learn joblib requests tqdm

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_candidates = {
    "Windows": ["Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"],
    "Darwin": ["Apple SD Gothic Neo", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"],
}.get(platform.system(), ["NanumGothic", "Noto Sans CJK KR"])

installed_fonts = {font.name for font in fm.fontManager.ttflist}
font_name = next((name for name in font_candidates if name in installed_fonts), None)

if font_name:
    plt.rcParams["font.family"] = font_name
    print(f"한글 폰트: {font_name}")
else:
    print("한글 폰트를 찾지 못했습니다. Windows는 맑은 고딕, macOS는 Apple SD 산돌고딕을 확인하세요.")

plt.rcParams["font.size"] = 12
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (6, 4)

**환경 확인 및 프로젝트 폴더 생성**

In [ ]:
import os, sys, json, re, shutil, zipfile, random, math, platform
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

# 원하는 저장 위치로 바꿀 수 있습니다.
# Windows 예: PROJECT_DIR = Path(r"D:\aihub_finance_project")
# macOS 예:   PROJECT_DIR = Path.home() / "Documents" / "aihub_finance_project"
PROJECT_DIR = (Path.cwd() / "aihub_finance_project").resolve()
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("운영체제:", platform.platform())
print("Python:", sys.version.split()[0])
print("프로젝트 폴더:", PROJECT_DIR)
print("데이터 폴더:", DATA_DIR)
print("디스크 여유 공간(GB):", round(shutil.disk_usage(PROJECT_DIR).free / 1024**3, 2))

assert sys.version_info >= (3, 10), "Python 3.10 이상이 필요합니다."
print("✔ 환경 확인 완료")

---

## **예제: 금융 이상거래 단계별 실습**

- 지도학습과 선형모델 · 분류모델
- 데이터: AI Hub **이상 판별을 위한 금융거래 정보 및 사용자 패턴 합성데이터**
  (`dataSetSn=71925`, 2025 구축, 2026-06 개방)
  - https://aihub.or.kr/aihubdata/data/view.do?currMenu=115&topMenu=100&dataSetSn=71925
  - 전자금융공동망 4,926,785건(이상 약 0.40%)
  - 카드거래 2,169,857건(이상 약 3.70%)
  - 본 실습은 라벨링 ZIP 2개(`TL_전자금융공동망.zip`, `TL_카드거래.zip`)만 사용한다.

- 실행 환경: Windows 10/11 또는 macOS의 Jupyter Notebook/JupyterLab/VS Code
- 실습 흐름:
  1. 로컬 폴더에서 ZIP 자동 탐색
  2. 파일이 없으면 AIHub API로 필요한 ZIP만 선택 다운로드
  3. ZIP을 모두 풀지 않고 분기별 CSV를 직접 읽기
  4. 결측·자료형·누수·불균형 처리
  5. Logistic Regression 학습과 F1·PR-AUC·ROC-AUC 평가
  6. 임계값·교차검증·규제·시간순 분할 비교

> AIHub에서 자동 다운로드하려면 해당 데이터셋의 이용 신청 승인이 완료되어 있어야 하며 개인 API Key가 필요하다.

### **[실습 1] 금융 이상거래여부 판별** (불균형 데이터 사용)

#### **0. 데이터 준비** — 로컬 ZIP을 우선 사용하고, 없으면 AIHub에서 다운로드

먼저 아래 위치에서 두 파일을 찾는다.

1. `aihub_finance_project/data/`
2. `aihub_finance_project/` 아래 모든 하위 폴더
3. 현재 노트북 폴더

직접 다운로드한 ZIP이 있다면 `data` 폴더에 넣으면 된다. 파일이 없을 때만 AIHub API Key 입력 창이 나타난다.

In [ ]:
import sklearn, pandas, matplotlib

print("scikit-learn:", sklearn.__version__)
print("pandas:", pandas.__version__)
print("matplotlib:", matplotlib.__version__)
print("✔ 라이브러리 준비 완료")

#### **0-1. 운영체제 독립형 AIHub 다운로드 함수 준비**

In [ ]:
import tarfile
import requests
from tqdm.auto import tqdm

AIHUB_BASE_URL = "https://api.aihub.or.kr"
AIHUB_DOWNLOAD_VERSION = "0.6"

def get_aihub_file_tree(dataset_key):
    url = f"{AIHUB_BASE_URL}/info/{dataset_key}.do"
    response = requests.get(url, timeout=60)
    response.encoding = response.apparent_encoding or "utf-8"
    text = response.text
    if ".zip" not in text.lower():
        raise RuntimeError(
            "AIHub 파일 목록을 받지 못했습니다. "
            f"(HTTP {response.status_code}) {text[:500]}"
        )
    return text

def download_with_resume(url, headers, target_path, chunk_size=8 * 1024**2):
    target_path = Path(target_path)
    partial_path = target_path.with_suffix(target_path.suffix + ".partial")
    downloaded = partial_path.stat().st_size if partial_path.exists() else 0
    request_headers = dict(headers)
    if downloaded:
        request_headers["Range"] = f"bytes={downloaded}-"

    with requests.get(url, headers=request_headers, stream=True, timeout=(30, 300)) as response:
        if 400 <= response.status_code < 500:
            response.raise_for_status()
        resume_ok = downloaded > 0 and response.status_code == 206
        if not resume_ok:
            downloaded = 0
        mode = "ab" if resume_ok else "wb"
        remaining = int(response.headers.get("Content-Length", 0))
        total = downloaded + remaining if remaining else None
        with partial_path.open(mode) as file, tqdm(
            total=total, initial=downloaded, unit="B", unit_scale=True,
            desc="AIHub 다운로드"
        ) as progress:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if chunk:
                    file.write(chunk)
                    progress.update(len(chunk))

    partial_path.replace(target_path)
    return target_path

def safe_extract_tar(tar_path, target_dir):
    target_dir = Path(target_dir).resolve()
    with tarfile.open(tar_path) as archive:
        for member in archive.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"TAR 링크 항목은 허용하지 않습니다: {member.name}")
            destination = (target_dir / member.name).resolve()
            if os.path.commonpath([str(target_dir), str(destination)]) != str(target_dir):
                raise RuntimeError(f"안전하지 않은 TAR 경로: {member.name}")
        archive.extractall(target_dir)

def merge_split_files(root_dir):
    groups = defaultdict(list)
    for part in Path(root_dir).rglob("*.part*"):
        match = re.search(r"\.part(\d+)$", part.name)
        if match:
            groups[part.parent / part.name[:match.start()]].append(
                (int(match.group(1)), part)
            )

    for merged_path, numbered_parts in groups.items():
        numbered_parts.sort(key=lambda item: item[0])
        print(f"분할 파일 병합: {merged_path.name} ({len(numbered_parts)}개)")
        with merged_path.open("wb") as merged:
            for _, part in numbered_parts:
                with part.open("rb") as source:
                    shutil.copyfileobj(source, merged, length=8 * 1024**2)
        for _, part in numbered_parts:
            part.unlink()

def download_aihub_files(dataset_key, filekeys, api_key, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    tar_path = target_dir / "download.tar"
    file_sn = ",".join(map(str, filekeys))
    url = f"{AIHUB_BASE_URL}/down/{AIHUB_DOWNLOAD_VERSION}/{dataset_key}.do?fileSn={file_sn}"
    download_with_resume(url, {"apikey": api_key}, tar_path)

    if not tarfile.is_tarfile(tar_path):
        with tar_path.open("rb") as file:
            message = file.read(1000).decode("utf-8", errors="replace")
        raise RuntimeError(f"AIHub 응답이 TAR 파일이 아닙니다. 승인/API Key를 확인하세요.\n{message}")

    print("TAR 해제 중...")
    safe_extract_tar(tar_path, target_dir)
    tar_path.unlink()
    merge_split_files(target_dir)
    return target_dir

print("✔ Windows/macOS 공통 다운로드 함수 준비 완료")

#### **0-2. 로컬 ZIP 탐색 및 필요한 경우 AIHub 선택 다운로드**

In [ ]:
from getpass import getpass

DATASET_KEY = "71925"
REQUIRED_ZIPS = {
    "bank": "TL_전자금융공동망.zip",
    "card": "TL_카드거래.zip",
}
MAX_TOTAL_DOWNLOAD_GB = 10.0

def find_local_zip(expected_name):
    roots = [DATA_DIR, PROJECT_DIR, Path.cwd()]
    seen = set()
    for root in roots:
        root = Path(root).resolve()
        if root in seen or not root.exists():
            continue
        seen.add(root)
        exact = [p for p in root.rglob("*.zip") if p.name == expected_name]
        if exact:
            return exact[0]
    return None

ZIP_BANK = find_local_zip(REQUIRED_ZIPS["bank"])
ZIP_CARD = find_local_zip(REQUIRED_ZIPS["card"])

if ZIP_BANK is None or ZIP_CARD is None:
    print("필요한 ZIP이 없어 AIHub 파일 목록을 조회합니다.")
    file_tree_text = get_aihub_file_tree(DATASET_KEY)

    size_factors = {"B": 1, "KB": 1024, "MB": 1024**2, "GB": 1024**3, "TB": 1024**4}

    def size_to_bytes(size_text):
        match = re.search(r"([\d.]+)\s*(B|KB|MB|GB|TB)", str(size_text), re.I)
        return np.nan if not match else float(match.group(1)) * size_factors[match.group(2).upper()]

    def parse_aihub_file_tree(text):
        rows = []
        for raw in text.splitlines():
            line = raw.strip()
            if "|" not in line or ".zip" not in line.lower():
                continue
            parts = [part.strip() for part in line.split("|")]
            if len(parts) < 3:
                continue
            filename = Path(re.sub(r"^[│├└─\s]+", "", parts[0]).strip()).name
            key_match = re.search(r"(\d+)", parts[2])
            if key_match:
                rows.append({
                    "filename": filename,
                    "size": parts[1],
                    "size_bytes": size_to_bytes(parts[1]),
                    "filekey": key_match.group(1),
                })
        return pd.DataFrame(rows)

    files_df = parse_aihub_file_tree(file_tree_text)
    assert len(files_df), "AIHub ZIP 파일 목록을 해석하지 못했습니다."

    def choose_finance_zip(keyword):
        candidates = files_df[
            files_df["filename"].str.contains(keyword, regex=False, na=False)
            & files_df["filename"].str.contains(r"(?i)TL", regex=True, na=False)
        ].copy()
        if candidates.empty:
            candidates = files_df[
                files_df["filename"].str.contains(keyword, regex=False, na=False)
            ].copy()
        assert len(candidates), f"'{keyword}' ZIP을 AIHub 목록에서 찾지 못했습니다."
        return candidates.sort_values("size_bytes", na_position="last").iloc[0]

    chosen_bank = choose_finance_zip("전자금융공동망")
    chosen_card = choose_finance_zip("카드거래")
    chosen = pd.DataFrame([chosen_bank, chosen_card])
    display(chosen[["filename", "size", "filekey"]])

    known_bytes = chosen["size_bytes"].fillna(0).sum()
    total_gb = known_bytes / 1024**3
    free_gb = shutil.disk_usage(PROJECT_DIR).free / 1024**3
    print(f"예상 다운로드 크기: {total_gb:.2f} GB")
    print(f"현재 디스크 여유: {free_gb:.2f} GB")
    assert total_gb <= MAX_TOTAL_DOWNLOAD_GB, (
        f"다운로드 크기가 안전 한도 {MAX_TOTAL_DOWNLOAD_GB}GB를 초과합니다. "
        "목록을 확인한 뒤 MAX_TOTAL_DOWNLOAD_GB를 조정하세요."
    )
    if known_bytes:
        assert shutil.disk_usage(PROJECT_DIR).free > known_bytes * 2.5, "다운로드·해제 공간이 부족합니다."

    AIHUB_API_KEY = getpass("AIHub API Key 입력: ").strip()
    assert len(AIHUB_API_KEY) >= 10, "API Key가 너무 짧습니다."
    download_aihub_files(
        dataset_key=DATASET_KEY,
        filekeys=chosen["filekey"].tolist(),
        api_key=AIHUB_API_KEY,
        target_dir=DATA_DIR,
    )

    ZIP_BANK = find_local_zip(REQUIRED_ZIPS["bank"])
    ZIP_CARD = find_local_zip(REQUIRED_ZIPS["card"])

assert ZIP_BANK is not None, f"{REQUIRED_ZIPS['bank']}을 찾지 못했습니다."
assert ZIP_CARD is not None, f"{REQUIRED_ZIPS['card']}을 찾지 못했습니다."
assert zipfile.is_zipfile(ZIP_BANK), f"올바른 ZIP이 아닙니다: {ZIP_BANK}"
assert zipfile.is_zipfile(ZIP_CARD), f"올바른 ZIP이 아닙니다: {ZIP_CARD}"

print("전자금융공동망 ZIP:", ZIP_BANK)
print("카드거래 ZIP      :", ZIP_CARD)
print("✔ 금융 라벨링 ZIP 준비 완료")

#### **0-3. ZIP을 풀지 않고 분기별 CSV를 읽는 함수**

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

def read_quarter(zip_path, year, quarter, **read_csv_kwargs):
    '''ZIP 안의 '<year>_<quarter>분기' CSV 한 장을 DataFrame으로 읽는다.

    코드값 '01', '00' 등의 앞자리 0을 보존하려고 먼저 문자열로 읽고,
    실제 수치형 변수만 이후 셀에서 명시적으로 변환한다.
    '''
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as archive:
        candidates = [
            name for name in archive.namelist()
            if f"{year}_{quarter}분기" in name and name.lower().endswith(".csv")
        ]
        if not candidates:
            csv_examples = [name for name in archive.namelist() if name.lower().endswith(".csv")][:10]
            raise FileNotFoundError(
                f"{zip_path.name}에서 {year}_{quarter}분기 CSV를 찾지 못했습니다. "
                f"CSV 예시: {csv_examples}"
            )
        name = candidates[0]
        for encoding in ("utf-8-sig", "cp949"):
            try:
                with archive.open(name) as file:
                    return pd.read_csv(
                        file, dtype=str, encoding=encoding, **read_csv_kwargs
                    )
            except UnicodeDecodeError:
                continue
    raise ValueError(f"CSV 인코딩을 확인하지 못했습니다: {name}")

def section(title):
    print("\n" + "=" * 100 + f"\n{title}\n" + "=" * 100)

print("✔ 분기별 CSV 읽기 함수 준비 완료")

#### **1. 데이터 읽기**: CSV 읽기와 구조 확인
- assert 문:뒤에 오는 조건이 참(True)이면 아무 일 없이 통과하고, 만약 거짓(False)이면 에러(AssertionError)를 발생시켜 프로그램을 그 자리에서 강제로 중단시킴

In [ ]:
bank = read_quarter(ZIP_BANK, 2024, 1)
print("shape:", bank.shape)                       # (41078, 12)
print("columns:", bank.columns.tolist())
print(bank.head(3).to_string())
print("\n✅ 결측치 수:\n", bank.isna().sum().to_string())
print("\n✅ target 분포:\n", bank["이상거래여부"].value_counts().to_string(), "\n→ 양성 %.2f%%" % (100 * (bank["이상거래여부"] == "1").mean()))
# 검증: 정답(label) 컬럼이 있고 두 값뿐이면 "지도학습용 라벨링 데이터"다
assert "이상거래여부" in bank.columns and set(bank["이상거래여부"].unique()) == {"0", "1"}
print("\n✅ ✔ 라벨(이상거래여부)이 있는 지도학습용 데이터")

#### **2. 데이터 확인**

- 자료형 정리 — 숫자처럼 보여도 '코드형 데이터'는 범주 데이터다

In [ ]:
bank["거래금액"] = pd.to_numeric(bank["거래금액"])
bank["거래시간대"] = pd.to_numeric(bank["거래시간대"])
bank["거래일자"] = pd.to_datetime(bank["거래일자"], format="%Y%m%d")
bank["요일"] = bank["거래일자"].dt.dayofweek.astype(str)              # 파생 feature (0=월 … 6=일) → 범주
bank["y"] = bank["이상거래여부"].astype(int)
print("\n✅ 데이터 타입:\n", bank.dtypes.to_string())
print("\n✅ 거래금액 요약:\n", bank["거래금액"].describe().round(0).to_string())
assert bank["거래금액"].dtype.kind in "if" and bank["거래일자"].dtype.kind == "M"
print("✅ ✔ 금액·시간대는 숫자, 코드(자금구분·매체구분·금융회사(계좌번호))는 문자열(범주)로 유지")

#### **3. 학습용 데이터 (X, y) 만들기**: 정답을 알려주는 컬럼(누수)과 식별자를 뺀다

In [ ]:
TARGET = "y"
LEAK = ["이상거래여부", "이상거래유형", "이상거래설명"]     # 정답 그 자체 · 정답이 있어야만 채워지는 컬럼
IDS = ["출금계좌일련번호", "입금계좌일련번호", "거래일자"]  # 식별자 · 원본 날짜(파생 feature 로 대체)
num_cols = ["거래금액", "거래시간대"]
cat_cols = ["출금금융회사일련번호", "입금금융회사일련번호", "자금구분", "매체구분", "요일"]
X = bank[num_cols + cat_cols]
y = bank[TARGET]
assert not any(c in X.columns for c in LEAK + IDS), "누수/식별자 컬럼이 X 에 남아 있다"
assert len(X) == len(y)
print("✅ X:", X.shape, "y:", y.shape, "| 양성 비율 %.2f%%" % (100 * y.mean()))
print("✅ ✔ AI Hub CSV → (X, y)  — 이 순간부터 '학습용 데이터'")

#### **4. 학습** : Pipeline 사용 → 간단 모델 → Accuracy 의 함정 확인
- LogisticRegression: 분류
    - max_iter=2000 (최대 반복 횟수 설정) : 모델이 최적의 정답(가중치)을 찾기 위해 알고리즘을 최대 반복 횟수
- DummyClassifier
    - [중요!] strategy="most_frequent"를 사용하여 가장 빈번하게 나타나는 클래스(정상 거래 '0')만을 예측하는 더미 모델을 만들어 정확도의 baseline을 만들고, 다른 모델과 비교하기 위함
- ColumnTransformer : 데이터 테이블의 열(Column)마다 서로 다른 전처리 과정을 각각 적용할 수 있도록 도와주는 도구
- Pipeline과 make_pipeline :
    - Pipeline과 (자동 이름 지정 방식), make_pipeline (자동 이름 지정 방식).
    - 데이터 전처리(Scaling, Imputation 등)부터 모델 학습(Classifier)까지의 모든 과정을 하나로 묶어주는 연결고리 역할을 하는 도구
    - 도구 모두 데이터 누수(Data Leakage)를 방지하고 코드를 간결하게 만든다는 최종 목적은 같지만, 작성하는 방식(문법)과 사용 편의성에서 차이가 있음    
- SimpleImputer : 데이터셋에 비어 있는 값, 즉 결측치(NaN, Null)를 특정 규칙에 따라 자동으로 채워주는 도구
-  train_test_split
    - stratify('계층화'):  stratify=y의 의미는 불균형 데이터셋이라도 X와 y의 비율을 정확하게 맞춰서 쪼개라는 의미.
    - 예: 사기 거래(1)가 10%, 정상 거래(0)가 90% → 학습 데이터(train)와 테스트 데이터(test) 모두 사기 거래 10% : 정상 거래 90%의 비율을 정확하게 맞춰서 쪼개라는 의미

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", min_frequency=20)), cat_cols),
])
clf = Pipeline([("pre", pre), ("model", LogisticRegression(max_iter=2000))]).fit(Xtr, ytr)
dummy = DummyClassifier(strategy="most_frequent").fit(Xtr, ytr)
model_acc = clf.score(Xte, yte)
dummy_acc = dummy.score(Xte, yte)
print("✅ Accuracy  Pipeline: %.4f   Dummy(전부 0): %.4f" % (model_acc, dummy_acc))
print("    → Test 양성 비율 %.2f%%: Accuracy만 보지 말고 F1·PR-AUC를 함께 확인해야 한다." % (100 * yte.mean()))
print("✅ 질문: '이 모델은 좋은 모델인가?' \n    → 분할 · 평가 지표 · 임계값 · 분류모델")

- [**결론**]
> 두 모델의 정확도가 비슷하다. 이는 ***불균형한 데이터셋에서 정확도만으로는 모델의 실제 성능을 제대로 파악하기 어렵다는 '정확도의 함정(Accuracy Trap)'을 보여주는 중요한 지표***가 된다. 즉, 모델이 실제로 이상 거래를 잘 감지하고 있는지 확인하기 위해, 단순히 다수 클래스만 맞추는 모델과 비교하는 것이 중요하다.



---



### **[실습 2] 카드 이상거래 단계별 실습**
— '_' 는 결측이다: 결측 표기를 통일하고 자료형을 나눈다

#### **1. 데이터 읽기 & 2. 데이터 확인**

In [ ]:
card = read_quarter(ZIP_CARD, 2024, 1, na_values=["_", ""])   # '_' 를 NaN 으로 읽는다
print("✅ shape:", card.shape)   # (12730, 32)
na = card.isna().sum()
print("\n✅ 결측이 있는 컬럼:\n", na[na > 0].to_string())
print("\n✅ 이상거래여부 분포:\n", card["이상거래여부"].value_counts().to_string(), "\n→ 양성 %.2f%%" % (100 * (card["이상거래여부"] == "1").mean()))
CARD_NUM = ["통합승인금액", "카드이용한도금액", "승인시간대", "연령", "경과일수_최종이용일자",
            "전월_매출건수", "전월_매출금액", "할부가능개월수"]
CARD_CAT = ["개인법인구분코드_회원", "국내해외여부", "가맹점광역시도코드", "남녀구분코드", "승인거래코드",
            "승인발생경로코드", "가맹점승인업종코드", "가맹점누적매출금액_구간화", "개인법인구분코드_가맹점",
            "가맹점여부_신규", "인터넷판매여부", "가맹점상태코드", "가맹점형태구분코드", "로고구분코드",
            "일시불할부구분코드", "카드구분코드"]
CARD_LEAK = ["이상거래여부", "이상거래유형", "이상거래설명"]
CARD_ID = ["카드KEY", "가맹점KEY", "승인SEQ", "기준년월", "승인일자"]
for c in CARD_NUM:
    card[c] = pd.to_numeric(card[c], errors="coerce")
card["승인일자"] = pd.to_datetime(card["승인일자"], format="%Y%m%d")

# 파생 컬럼
card["요일"] = card["승인일자"].dt.dayofweek.astype(str)
card["log_금액"] = np.log1p(card["통합승인금액"].clip(lower=0))       # 치우친 금액 → 로그 (수치 파생)

CARD_NUM += ["log_금액"]; CARD_CAT += ["요일"]
assert set(CARD_NUM + CARD_CAT + CARD_LEAK + CARD_ID) == set(card.columns), \
    set(card.columns) - set(CARD_NUM + CARD_CAT + CARD_LEAK + CARD_ID)
print("\n✅ ✔ 32 + 2(파생) 컬럼을 수치 %d · 범주 %d · 누수 %d · 식별자 %d 로 빠짐없이 분류" %
      (len(CARD_NUM), len(CARD_CAT), len(CARD_LEAK), len(CARD_ID)))

#### **3. 학습용 데이터 (X, y) 만들기**:

In [ ]:
Xc = card[CARD_NUM + CARD_CAT]
yc = (card["이상거래여부"] == "1").astype(int)

#### **4. 학습** : Pipeline(class_weight) → 분류 지표
- max_iter=5000 (최대 반복 횟수 설정) :
    - 모델이 최적의 정답(가중치)을 찾기 위해 알고리즘을 최대 반복 횟수
- class_weight="balanced" (데이터 분균형 해결):
    - 데이터셋에서 양이 적은 클래스(타깃)에 더 큰 가중치(중요도)를 자동으로 부여하는 설정

> "데이터 양이 불균형하더라도 소수 데이터를 소홀히 하지 않고(class_weight="balanced"), 컴퓨터가 지치지 않고 끝까지 정답을 찾을 수 있도록 넉넉하게 반복 학습(max_iter=5000)해라"라는 뜻

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, average_precision_score, balanced_accuracy_score)

Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)
pre_card = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), CARD_NUM),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore", min_frequency=20)), CARD_CAT),
])


def card_pipe(**kw):
    return Pipeline([("pre", pre_card), ("model", LogisticRegression(max_iter=5000, **kw))])


results = {}
for name, m in {"Dummy": DummyClassifier(strategy="most_frequent"),
                "LR(basic)": card_pipe(),
                "LR(balanced)": card_pipe(class_weight="balanced")}.items():
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    prob = m.predict_proba(Xte)[:, 1]
    results[name] = dict(Acc=m.score(Xte, yte), BalAcc=balanced_accuracy_score(yte, p),
                         Prec=precision_score(yte, p, zero_division=0), Rec=recall_score(yte, p),
                         F1=f1_score(yte, p), ROC_AUC=roc_auc_score(yte, prob), PR_AUC=average_precision_score(yte, prob))
res = pd.DataFrame(results).T.round(3)
print(res.to_string(justify='right'))

print("\n✅ LR(balanced) confusion matrix [[TN FP][FN TP]]:\n", confusion_matrix(yte, card_pipe(class_weight="balanced").fit(Xtr, ytr).predict(Xte)))
rec_basic = res.loc["LR(basic)", "Rec"]
rec_balanced = res.loc["LR(balanced)", "Rec"]
if rec_balanced > rec_basic:
    print("\n✅ 이 분할에서는 class_weight='balanced'가 Recall을 높였다. Precision과 함께 비교한다.")
else:
    print("\nℹ 이 분할에서는 Recall이 증가하지 않았다. class_weight의 효과는 데이터와 분할에 따라 달라질 수 있다.")



---



### **[실습 3] 분할 비율과 seed 의 영향**
— 같은 모델·같은 데이터인데 점수가 흔들린다

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
split_rows = []
for ratio in [0.1, 0.2, 0.3, 0.5]:
    f1s = []                        # f1 score 값
    for seed in range(10):
        Xtr, Xte, ytr, yte = train_test_split(Xc, yc, test_size=ratio, stratify=yc, random_state=seed)
        f1s.append(f1_score(yte, card_pipe(class_weight="balanced").fit(Xtr, ytr).predict(Xte)))
    split_rows.append(dict(test_size=ratio, n_test=len(Xte), F1_mean=np.mean(f1s), F1_std=np.std(f1s)))
split_df = pd.DataFrame(split_rows).round(3)
print(split_df.to_string(index=False))
print("✅ ✔ 한 번의 seed 점수는 우연을 포함한다 → 평균 ± 표준편차로 보면 더 효과적")



---



### **[실습 4] Stratified K-fold** — 점수의 분포, Train–Test gap
- StratifiedKFold:
    - 데이터를 분할할 때 각 폴드(Fold) 안에 포함된 타겟(정답) 클래스의 비율이 전체 데이터의 비율과 똑같아지도록 보장하며 쪼개는 알고리즘
    - n_splits=5 : 전체 데이터를 5개의 덩어리(폴드)로 나눈다는 뜻. 모델 학습과 검증을 총 5번 반복하게 됨
- cross_validate:
    - cross_val_score는 모델의 성능을 교차 검증(Cross Validation)으로 평가하여 검증 점수(Score)들의 배열만 깔끔하게 반환하는 가장 기본적인 함수
    - 기존의 cross_val_score보다 확장된 기능을 제공하는 함수, 여러 개의 평가지표(scoring=[...])를 동시에 계산할 수 있음
- scoring=["f1", "average_precision", "roc_auc"] :
    - 불균형 데이터셋 평가에 가장 중요한 3가지 핵심 평가지표를 한 번에 계산
    - f1: 정밀도(Precision)와 재현율(Recall)의 조화평균으로, 불균형 분류에서 정확도(Accuracy)보다 훨씬 신뢰할 수 있는 지표
    - average_precision (AP): 정밀도-재현율 곡선(PR Curve) 아래의 면적으로, 데이터가 불균형할 때 모델이 얼마나 소수 클래스를 잘 잡아내는지 보여주는 아주 강력한 지표
    - roc_auc: 모델의 전반적인 분류 정확도를 나타내며, 대략적인 이진 분류 성능을 파악할 때 사용
- return_train_score=True
    - 검증 데이터의 점수뿐만 아니라, 학습(Train)할 때 사용한 데이터의 점수도 결과에 포함하라는 뜻

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cvres = cross_validate(card_pipe(class_weight="balanced"), Xc, yc, cv=cv,
                       scoring=["f1", "average_precision", "roc_auc"], return_train_score=True)
print("✅ fold F1      :", cvres["test_f1"].round(3), "mean=%.3f std=%.3f" % (cvres["test_f1"].mean(), cvres["test_f1"].std()))
print("✅ train F1     : %.3f   test F1: %.3f   gap: %.3f" % (cvres["train_f1"].mean(), cvres["test_f1"].mean(),
                                                            cvres["train_f1"].mean() - cvres["test_f1"].mean()))
print("✅ PR-AUC       : %.3f    ROC-AUC: %.3f" % (cvres["test_average_precision"].mean(), cvres["test_roc_auc"].mean()))
print("✅ ✔ 전처리가 Pipeline 안에 있으므로 fold 마다 Scaler·Encoder 가 새로 fit → 누수 없음")



---



### [실습 5] 임계값은 hyperparameter — 0.5 가 아니라 비용에 맞게 CV 로 고른다
- TunedThresholdClassifierCV
    - (scikit-learn 1.5 버전 이상)에 새롭게 도입된, 이진 분류 모델의 결정 임계값(Decision Threshold)을 교차 검증을 통해 자동으로 최적화해 주는 메타 추정기(Meta-estimator)
    - 예: "몇 %부터 1(정상/사기 등)로 판정할 것인가?"의 기준선(기본값 50%)을 컴퓨터가 교차 검증을 통해 가장 똑똑한 수치로 알아서 찾아주는 도구
- NumPy 임계값 목록으로 F1을 직접 계산
    - scikit-learn 세부 버전에 관계없이 실행할 수 있다.
    - Test로 최적 임계값을 고르면 누수이므로 교육용 비교에만 사용한다.

In [ ]:
from sklearn.model_selection import TunedThresholdClassifierCV

plain = card_pipe(class_weight="balanced").fit(Xtr, ytr)
tuned = TunedThresholdClassifierCV(
    card_pipe(class_weight="balanced"), scoring="f1", cv=5
).fit(Xtr, ytr)

prob = plain.predict_proba(Xte)[:, 1]
plain_pred = plain.predict(Xte)
tuned_pred = tuned.predict(Xte)

print("✅ 임계값 0.5   : F1 %.3f  Prec %.3f  Rec %.3f" % (
    f1_score(yte, plain_pred),
    precision_score(yte, plain_pred, zero_division=0),
    recall_score(yte, plain_pred, zero_division=0),
))
print("✅ CV 최적 %.3f : F1 %.3f  Prec %.3f  Rec %.3f" % (
    tuned.best_threshold_,
    f1_score(yte, tuned_pred),
    precision_score(yte, tuned_pred, zero_division=0),
    recall_score(yte, tuned_pred, zero_division=0),
))

# Test 점수로 임계값을 고르면 누수다. 아래 계산은 교육용 비교에만 사용한다.
thresholds = np.linspace(0.01, 0.99, 99)
test_f1s = np.array([
    f1_score(yte, (prob >= threshold).astype(int), zero_division=0)
    for threshold in thresholds
])
best_index = int(test_f1s.argmax())
print("✅ (참고) Test를 직접 훑은 F1 최대 %.3f @ %.2f — 이 값으로 모델을 선택하면 누수" % (
    test_f1s[best_index], thresholds[best_index]
))

for threshold in [0.3, 0.5, 0.7, 0.9]:
    pred = (prob >= threshold).astype(int)
    print("   thr %.1f → Prec %.3f  Rec %.3f  F1 %.3f  (양성 예측 %d건)" % (
        threshold,
        precision_score(yte, pred, zero_division=0),
        recall_score(yte, pred, zero_division=0),
        f1_score(yte, pred, zero_division=0),
        pred.sum(),
    ))

print("✅ FN 비용이 큰 문제는 낮은 임계값, FP 비용이 큰 문제는 높은 임계값을 검토")

- **[결론]**
    - **FN 비용이 큰 문제(사기·질병)는 낮은 임계값**
    > FN(False Negative)이란?: 실제로는 사건(사기, 질병)이 발생했는데, 모델이 "정상(아님)"이라고 잘못 예측한 경우
    >> - 확률이 0.5가 안 되더라도, "조금이라도 사기나 질병일 확률이 있다면(예: 0.2 이상이면) 무조건 1(위험)로 의심하고 검사해라!"라는 뜻
    >> - 기준(임계값)을 낮추면 애꿎은 정상인도 재검사를 받게 될 수 있지만(FP 증가), 진짜 위험한 놈을 놓치는 치명적인 상황(FN)을 완벽히 방지할 수 있
    - **FP 비용이 큰 문제(스팸·차단)는 높은 임계값**
    > FP(False Positive)이란?: 실제로는 정상인데, 모델이 "문제(스팸, 차단 대상)"라고 잘못 예측한 경우
    >> - "확률이 0.5 수준인 건 함부로 스팸이나 해커로 단정 짓지 말고, 0.8~0.9처럼 90% 이상 확실할 때만 1(스팸/차단)로 처리해라!"라는 뜻
    >> - 스팸 메일이 몇 개 편지함에 들어오는 불편함(FN 증가)을 감수하더라도, 중요한 메일이 삭제되는 대참사(FP)를 막기 위해 기준선을 깐깐하게 올리는 것


---



### **[실습 6] 로지스틱 회귀 계수 해석**

- exp(w) = odds ratio (표준화된 1σ 또는 범주 1개 기준)    
    - 예: **"odds_ratio=exp(w)": np.exp(top).round(2)**
    - 머신러닝 가중치인 w (회귀계수)에 자연상수 e의 지수(Exponential)를 취해준 값(np.exp(w))을 뜻한다.
        
    > - 이 코드는 로지스틱 회귀(Logistic Regression) 모델에서 각 변수가 타겟 변수(예: 사기 거래, 질병 등 양성 클래스)의 발생 확률에 미치는 영향력을 오즈비(Odds Ratio, 승산비)라는 직관적인 지표로 변환하는 코드
    > - 로지스틱 회귀 모델은 내부적으로 확률을 바로 예측하는 것이 아니라 회귀계수 w들을 로그 오즈(Log Odds) 형태로 학습함. 그래서 해석하기 위해 반대로 지수함수 적용함

- intercept b = -7.295 → 기준 확률 sigmoid(b) = 0.001 해석
    - intercept b는 수학에서 말하는 Y절편(기본값
    - 이를 확률로 변환해 주는 시그모이드(Sigmoid) 함수에 넣었더니 0.001이 나온 것임
> **의미**: 데이터의 모든 조건이 완전히 평균이거나 아무런 특징이 없는 '기본 상태(아무것도 모르는 상태)'일 때, 사건(사기/질병 등)이 발생할 확률은 겨우 0.1%(0.001)라는 뜻

In [ ]:
model = plain.named_steps["model"]
names = plain.named_steps["pre"].get_feature_names_out()
coef = pd.Series(model.coef_[0], index=names)
top = pd.concat([coef.sort_values().head(6), coef.sort_values().tail(6)])

# pd.DataFrame({"w": top.round(3), "odds_ratio=exp(w)": np.exp(top).round(2)})
print(pd.DataFrame({"w": top.round(3), "odds_ratio=exp(w)": np.exp(top).round(2)}).to_string(justify='right'))
print("\n✅ intercept b = %.3f → 기준 확률 sigmoid(b) = %.3f" % (model.intercept_[0], 1 / (1 + np.exp(-model.intercept_[0]))))
assert len(coef) == model.coef_.shape[1]



---



### **[실습 7]  로지스틱 회귀 모델의 성능을 극대화**
- 로지스틱 회귀 모델의 성능을 극대화하기 위해 **규제(Regularization) 방식(L1, L2)**과 **규제 강도(C)**를 바꿔가며 최적의 하이퍼파라미터를 찾는 과정(Grid Search)
- 계수가 0 이 되는 것을 본다 (C = 1/alpha, 작을수록 강한 규제)
- LogisticRegression(C=C, l1_ratio=l1_ratio, solver=solver,class_weight="balanced", max_iter=5000)
    - C=C (규제 강도의 역수)
        - 모델이 복잡해지는 것을 막아주는 제동 장치.
        - 주의할 점은 숫자가 작을수록 규제가 강해지고, 크면 규제가 약해진다
    - l1_ratio=l1_ratio  (L1 규제 비율)
        - ElasticNet 규제를 쓸 때 L1 규제를 얼마나 섞을지 정하는 비율
        - 이 코드는 코드의 2번째 줄 루프[("L2", 0.0, "lbfgs"), ("L1", 1.0, "liblinear")] 를 통해 0.0일 때는 순수 L2 규제, 1.0일 때는 순수 L1 규제가 되도록 설정된 상태
    - solver=solver (최적화 알고리즘)
        - 모델이 정답을 찾기 위해 수학적 계산을 수행하는 내부 엔진
        - lbfgs 엔진은 기본 엔진으로 L2 규제에 최적화되어 있고, liblinear 엔진은 데이터가 적을 때 유용하며 L1 규제를 지원함
- **L1 규제 (Lasso 원리): "쓸모없는 변수는 다 쳐낸다!"**
    - 특징: 중요하지 않다고 생각하는 변수의 가중치(w)를 정확히 0으로 만듦
    - 비유: "중요한 변수 몇 개만 골라서 컴팩트하게 예측하겠다"는 변수 선택(Feature Selection) 효과를 냅니다.
- **L2 규제 (Ridge 원리): "다 같이 힘을 빼고 골고루 쓰자!"**
    - 특징: 변수의 가중치를 완전히 0으로 만들지는 않고, 골고루 숫자의 크기(절대값)만 작게 축소시킴
    - 비유: "특정 변수 하나에만 의존하지 말고, 모든 변수를 얌전하게 만들어서 골고루 반영하겠다"는 방식

In [ ]:
rows = []
for penalty, l1_ratio, solver in [("L2", 0.0, "lbfgs"), ("L1", 1.0, "liblinear")]:
    for C in [0.001, 0.01, 0.1, 1, 10]:
        pipe = Pipeline([("pre", pre_card), ("model", LogisticRegression(C=C, l1_ratio=l1_ratio, solver=solver,
                                                                         class_weight="balanced", max_iter=5000))])
        s = cross_val_score(pipe, Xtr, ytr, cv=cv, scoring="average_precision")
        pipe.fit(Xtr, ytr)
        w = pipe.named_steps["model"].coef_[0]
        rows.append(dict(penalty=penalty, C=C, CV_PR_AUC=s.mean(), n_features=len(w),
                         n_zero=int((np.abs(w) < 1e-6).sum()), max_abs_w=np.abs(w).max()))
reg_df = pd.DataFrame(rows).round(3)
print(reg_df.to_string(index=False))
print("✅ ✔ L1 은 계수를 정확히 0 으로(변수 선택), L2 는 작게 축소 — 회귀의 Lasso/Ridge 와 같은 원리")

- [결론]
    - **L1 규제 (Lasso 원리): "쓸모없는 변수는 다 쳐낸다!"**
        - 특징: 중요하지 않다고 생각하는 변수의 가중치(w)를 정확히 0으로 만듦
        - 비유: "중요한 변수 몇 개만 골라서 컴팩트하게 예측하겠다"는 변수 선택(Feature Selection) 효과를 냅니다.
    - **L2 규제 (Ridge 원리): "다 같이 힘을 빼고 골고루 쓰자!"**
        - 특징: 변수의 가중치를 완전히 0으로 만들지는 않고, 골고루 숫자의 크기(절대값)만 작게 축소시킴
        - 비유: "특정 변수 하나에만 의존하지 말고, 모든 변수를 골고루 반영하겠다"는 방식

----

### **[실습 8] 시간순 분할** — 과거(2023)로 학습해 미래(2024)를 맞히면 점수가 얼마나 떨어지는가?

In [ ]:
def load_card_year(year):
    frames = []
    for q in [1, 2, 3, 4]:
        d = read_quarter(ZIP_CARD, year, q, na_values=["_", ""])
        frames.append(d)
    d = pd.concat(frames, ignore_index=True)
    for c in CARD_NUM:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    d["승인일자"] = pd.to_datetime(d["승인일자"], format="%Y%m%d")
    d["요일"] = d["승인일자"].dt.dayofweek.astype(str)
    d["log_금액"] = np.log1p(d["통합승인금액"].clip(lower=0))
    return d[CARD_NUM + CARD_CAT], (d["이상거래여부"] == "1").astype(int)


X23, y23 = load_card_year(2023)
X24, y24 = load_card_year(2024)
m_time = card_pipe(class_weight="balanced").fit(X23, y23)             # 과거 → 미래
p24 = m_time.predict_proba(X24)[:, 1]
Xall, yall = pd.concat([X23, X24]), pd.concat([y23, y24])
a, b, c, d = train_test_split(Xall, yall, test_size=len(X24) / len(Xall), stratify=yall, random_state=42)
m_rand = card_pipe(class_weight="balanced").fit(a, c)                 # 무작위 섞기
prand = m_rand.predict_proba(b)[:, 1]
print("✅ 학습 2023 %d건 → 테스트 2024 %d건" % (len(X23), len(X24)))
print("✅ 시간순 분할  : PR-AUC %.3f  ROC-AUC %.3f  F1 %.3f" % (average_precision_score(y24, p24), roc_auc_score(y24, p24), f1_score(y24, (p24 >= 0.5).astype(int))))
print("✅ 무작위 분할  : PR-AUC %.3f  ROC-AUC %.3f  F1 %.3f" % (average_precision_score(d, prand), roc_auc_score(d, prand), f1_score(d, (prand >= 0.5).astype(int))))
print("    ※ 이 데이터는 합성 데이터라 연도 간 분포 변화(drift)가 거의 없어 두 점수가 비슷하다 — 실데이터에서는 보통 시간순 점수가 더 낮다")


---

### **[실습 9] 이상거래인지 판정하는 웹 화면**

지금까지는 **지표**로 모델을 평가했다. 여기서는 **학습에 한 번도 쓰이지 않은 거래**를
목록에서 하나 골라 "이 건이 이상거래인가?"를 직접 판정해 본다.

#### **9-1. 서비스용 모델 다시 학습 + 판정 임계값 결정**

- 앞 셀들에서 `pre`·`pre_card` 객체는 여러 번 `fit` 되었다(마지막 `fit` 상태만 남아 있다).
  헷갈리지 않도록 `clone()` 으로 전처리기를 복제해 **서비스용 파이프라인을 새로 학습**한다.
- 분할은 본문과 같은 `random_state=42` 이므로 **학습에 쓰인 행도 본문과 정확히 같다.**
- 두 모델 모두 `class_weight="balanced"` 를 쓴다. 양성 비율이 0.40% · 3.70% 로 낮아
  기본 설정은 거의 모든 건을 '정상'이라고만 답하기 때문이다.
- 판정 임계값은 **Train 안에서만** 교차검증(out-of-fold) 확률로 F1이 가장 높은 값을 고른다.
  Test·holdout 을 보고 고르면 누수다 → [실습 5]

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score
import joblib

SERVE_DIR = OUTPUT_DIR / "serve"
SERVE_DIR.mkdir(parents=True, exist_ok=True)

# 1) 본문과 같은 random_state → 학습/평가 분할이 정확히 같다
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

# 2) 전처리기를 clone 해서 서비스용 파이프라인을 새로 학습한다
bank_model = Pipeline([
    ("pre", clone(pre)),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
]).fit(Xb_tr, yb_tr)

card_model = Pipeline([
    ("pre", clone(pre_card)),
    ("model", LogisticRegression(max_iter=5000, class_weight="balanced")),
]).fit(Xc_tr, yc_tr)


# 3) 임계값은 Train 안에서 out-of-fold 확률로만 고른다 (Test·holdout 을 보지 않는다)
def pick_threshold(model, X_train, y_train, n_splits=3):
    folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = cross_val_predict(clone(model), X_train, y_train, cv=folds, method="predict_proba")[:, 1]
    grid = np.linspace(0.05, 0.95, 91)
    scores = np.array([f1_score(y_train, (oof >= t).astype(int), zero_division=0) for t in grid])
    best = int(scores.argmax())
    return float(grid[best]), float(scores[best])


BANK_THR, bank_oof_f1 = pick_threshold(bank_model, Xb_tr, yb_tr)
CARD_THR, card_oof_f1 = pick_threshold(card_model, Xc_tr, yc_tr)

print("✅ 전자금융공동망 : Train %6d건 (양성 %.2f%%) → 임계값 %.2f (out-of-fold F1 %.3f)"
      % (len(Xb_tr), 100 * yb_tr.mean(), BANK_THR, bank_oof_f1))
print("✅ 카드거래       : Train %6d건 (양성 %.2f%%) → 임계값 %.2f (out-of-fold F1 %.3f)"
      % (len(Xc_tr), 100 * yc_tr.mean(), CARD_THR, card_oof_f1))

joblib.dump({"model": bank_model, "threshold": BANK_THR}, SERVE_DIR / "bank_logreg.joblib")
joblib.dump({"model": card_model, "threshold": CARD_THR}, SERVE_DIR / "card_logreg.joblib")

assert 0.0 < BANK_THR < 1.0 and 0.0 < CARD_THR < 1.0
print("✅ 모델 저장 :", SERVE_DIR)
print("✅ ✔ 서비스용 모델 준비 완료")

#### **9-2. 학습에 참여하지 않은 거래 준비**

- **2024년 2분기 CSV**를 읽는다.
  - 전자금융공동망: 본문은 2024년 1분기만 사용했으므로 2분기는 통째로 미학습 데이터다.
  - 카드거래: [실습 8]에서 2023·2024년 전체를 읽었지만 **서비스용 모델은 2024년 1분기의 Train 분할로만**
    학습했으므로, 2분기 거래는 이 모델이 처음 보는 거래다.
- 분기 CSV를 찾지 못하면 1분기의 **Test 분할**(학습에 쓰이지 않은 20%)로 자동 대체한다.
- 이상거래는 워낙 드물어(0.4%) 목록을 무작위로 채우면 한 건도 안 보인다.
  그래서 웹 화면에 띄울 **표본은 이상 건과 정상 건을 섞어서** 만든다.
  (지표 계산은 표본이 아니라 holdout 전체로 한다 → 9-3)

In [ ]:
BANK_NUM = list(dict.fromkeys(num_cols))
BANK_CAT = list(dict.fromkeys(cat_cols))
CARD_NUM_U = list(dict.fromkeys(CARD_NUM))       # 셀을 여러 번 실행해 중복된 경우를 대비
CARD_CAT_U = list(dict.fromkeys(CARD_CAT))

HOLDOUT_YEAR, HOLDOUT_QUARTER = 2024, 2          # 학습에 쓰지 않은 분기
POOL_FRAUD, POOL_NORMAL = 40, 160                # 웹 목록에 띄울 표본 수 (이상 / 정상)


def prep_bank(df):
    """전자금융공동망 원본 CSV → 2번 셀과 똑같은 자료형·파생 컬럼"""
    df = df.copy()
    df["거래금액"] = pd.to_numeric(df["거래금액"], errors="coerce")
    df["거래시간대"] = pd.to_numeric(df["거래시간대"], errors="coerce")
    df["거래일자"] = pd.to_datetime(df["거래일자"], format="%Y%m%d", errors="coerce")
    df["요일"] = df["거래일자"].dt.dayofweek.astype(str)
    df["y"] = (df["이상거래여부"] == "1").astype(int)
    return df


def prep_card(df):
    """카드거래 원본 CSV → [실습 2]와 똑같은 자료형·파생 컬럼"""
    df = df.copy()
    for c in CARD_NUM_U:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["승인일자"] = pd.to_datetime(df["승인일자"], format="%Y%m%d", errors="coerce")
    df["요일"] = df["승인일자"].dt.dayofweek.astype(str)
    df["log_금액"] = np.log1p(df["통합승인금액"].clip(lower=0))
    df["y"] = (df["이상거래여부"] == "1").astype(int)
    return df


def load_holdout(kind):
    """미학습 거래를 읽는다. 분기 CSV가 없으면 Test 분할로 대체한다."""
    quarter_label = f"{HOLDOUT_YEAR}년 {HOLDOUT_QUARTER}분기 전체 (학습에 쓰지 않은 분기)"
    try:
        if kind == "bank":
            return prep_bank(read_quarter(ZIP_BANK, HOLDOUT_YEAR, HOLDOUT_QUARTER)), quarter_label
        return prep_card(read_quarter(ZIP_CARD, HOLDOUT_YEAR, HOLDOUT_QUARTER,
                                      na_values=["_", ""])), quarter_label
    except FileNotFoundError as exc:
        print("ℹ 분기 CSV를 찾지 못해 Test 분할로 대체합니다:", exc)
        base = bank if kind == "bank" else card
        index = Xb_te.index if kind == "bank" else Xc_te.index
        fallback = base.loc[index].copy()
        fallback["y"] = (fallback["이상거래여부"] == "1").astype(int)
        return fallback, "2024년 1분기 Test 분할 (학습에 쓰지 않은 20%)"


def fmt_value(v):
    """화면에 바로 뿌릴 수 있는 문자열로 바꾼다."""
    if v is None or (not isinstance(v, str) and pd.isna(v)):
        return "-"
    if isinstance(v, pd.Timestamp):
        return v.strftime("%Y-%m-%d")
    if isinstance(v, (bool, np.bool_)):
        return str(bool(v))
    if isinstance(v, (int, np.integer)):
        return f"{int(v):,}"
    if isinstance(v, (float, np.floating)):
        return f"{v:,.0f}" if abs(v) >= 1000 else f"{v:,.3f}".rstrip("0").rstrip(".")
    return str(v)


def make_pool(entry, n_fraud=POOL_FRAUD, n_normal=POOL_NORMAL, seed=42):
    """이상 건과 정상 건을 섞어 웹 목록용 표본 인덱스를 만든다."""
    rng = np.random.RandomState(seed)
    labels = entry["y"].to_numpy()
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    take_f = rng.choice(fraud_idx, size=min(n_fraud, len(fraud_idx)), replace=False)
    take_n = rng.choice(normal_idx, size=min(n_normal, len(normal_idx)), replace=False)
    pool = np.concatenate([take_f, take_n])
    rng.shuffle(pool)
    return pool.astype(int)


def build_entry(key, name, model, threshold, feat_cols, list_cols, label_cols):
    raw, source = load_holdout(key)
    raw = raw.reset_index(drop=True)
    entry = {
        "key": key,
        "name": name,
        "model": model,
        "threshold": float(threshold),
        "raw": raw,
        "source": source,
        "X": raw[feat_cols],
        "y": raw["y"].astype(int),
        "feat_cols": feat_cols,
        "list_cols": list_cols,
        "label_cols": label_cols,      # 정답 설명용(모델 입력 아님)
    }
    entry["pool"] = make_pool(entry)
    return entry


FIN_DATASETS = {
    "bank": build_entry(
        "bank", "전자금융공동망", bank_model, BANK_THR,
        feat_cols=BANK_NUM + BANK_CAT,
        list_cols=["거래일자", "거래금액", "거래시간대", "자금구분", "매체구분",
                   "출금금융회사일련번호", "입금금융회사일련번호"],
        label_cols=["이상거래유형", "이상거래설명"],
    ),
    "card": build_entry(
        "card", "카드거래", card_model, CARD_THR,
        feat_cols=CARD_NUM_U + CARD_CAT_U,
        list_cols=["승인일자", "통합승인금액", "승인시간대", "가맹점승인업종코드",
                   "국내해외여부", "인터넷판매여부", "연령"],
        label_cols=["이상거래유형", "이상거래설명"],
    ),
}

for key, entry in FIN_DATASETS.items():
    print("✅ %s : %s" % (entry["name"], entry["source"]))
    print("   미학습 거래 %d건 (이상 %d건, %.2f%%) · 웹 목록 표본 %d건"
          % (len(entry["y"]), int(entry["y"].sum()), 100 * entry["y"].mean(), len(entry["pool"])))
    assert list(entry["X"].columns) == entry["feat_cols"], "모델 입력 컬럼이 학습 때와 다르다"

display(FIN_DATASETS["bank"]["raw"].loc[FIN_DATASETS["bank"]["pool"][:5],
                                        FIN_DATASETS["bank"]["list_cols"] + ["이상거래여부"]])
print("✅ ✔ 미학습 거래 준비 완료")

#### **9-3. 거래 1건 판정 함수 + holdout 전체 일괄 평가**

- `predict_transaction(kind, row_id)` : 거래 1건 → **이상거래 확률 · 판정 · 판정 근거**
- 로지스틱 회귀는 선형 모델이므로 판정 근거를 그대로 분해할 수 있다.

  `logit = 절편 b + Σ(표준화된 값 z × 계수 w)` → `확률 p = sigmoid(logit)`

  각 항 `z × w` 가 **양수면 '이상' 쪽으로, 음수면 '정상' 쪽으로** 이 거래를 밀었다는 뜻이다.
  → [실습 6]의 계수 해석을 거래 1건에 적용한 것
- holdout 전체 점수는 Test 점수와 비교한다. 비슷하면 일반화가 된 것이고, 크게 낮으면 과적합을 의심한다.

In [ ]:
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)


def clean_code(v):
    """'2.0' 처럼 실수로 읽힌 코드값을 '2' 로 다듬는다."""
    s = fmt_value(v)
    return s[:-2] if s.endswith(".0") else s


def transform_row(entry, row_id):
    """전처리 파이프라인을 통과한 1행 벡터와 feature 이름"""
    pre_step = entry["model"].named_steps["pre"]
    z = pre_step.transform(entry["X"].iloc[[int(row_id)]])
    z = z.toarray() if hasattr(z, "toarray") else np.asarray(z)
    return z[0], pre_step.get_feature_names_out()


def predict_transaction(kind, row_id, threshold=None, top_k=6):
    """거래 1건 → 확률·판정·근거 dict"""
    entry = FIN_DATASETS[kind]
    row_id = int(row_id)
    thr = entry["threshold"] if threshold is None else float(threshold)

    prob = float(entry["model"].predict_proba(entry["X"].iloc[[row_id]])[0, 1])
    z, names = transform_row(entry, row_id)
    model = entry["model"].named_steps["model"]
    weights = model.coef_[0]
    contrib = z * weights
    order = np.argsort(-np.abs(contrib))[:top_k]

    raw_row = entry["raw"].iloc[row_id]
    truth = int(entry["y"].iloc[row_id])

    return {
        "kind": kind,
        "row_id": row_id,
        "prob": prob,
        "threshold": thr,
        "pred": int(prob >= thr),
        "true": truth,
        "correct": int(prob >= thr) == truth,
        "fraud_type": clean_code(raw_row.get("이상거래유형")) if truth else "-",
        "fraud_desc": fmt_value(raw_row.get("이상거래설명")) if truth else "-",
        "intercept": float(model.intercept_[0]),
        "logit": float(model.intercept_[0] + contrib.sum()),
        "contributions": [
            {
                "name": str(names[i]).split("__", 1)[-1],
                "z": float(z[i]),
                "w": float(weights[i]),
                "c": float(contrib[i]),
            }
            for i in order
        ],
        "fields": [{"k": c, "v": fmt_value(raw_row[c])} for c in entry["feat_cols"]],
    }


# holdout 전체 일괄 평가 (확률은 한 번만 계산해 두고 웹 화면에서 재사용한다)
for key, entry in FIN_DATASETS.items():
    entry["prob_all"] = entry["model"].predict_proba(entry["X"])[:, 1]
    y_true = entry["y"].to_numpy()
    pred = (entry["prob_all"] >= entry["threshold"]).astype(int)

    entry["metrics"] = {
        "n": int(len(y_true)),
        "n_pos": int(y_true.sum()),
        "threshold": entry["threshold"],
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, entry["prob_all"])),
        "pr_auc": float(average_precision_score(y_true, entry["prob_all"])),
    }
    m = entry["metrics"]

    section(f"{entry['name']} — 미학습 {m['n']:,}건 일괄 평가  ({entry['source']})")
    print("✅ 임계값 %.2f : Prec %.3f  Rec %.3f  F1 %.3f   | PR-AUC %.3f  ROC-AUC %.3f"
          % (m["threshold"], m["precision"], m["recall"], m["f1"], m["pr_auc"], m["roc_auc"]))
    pred_half = (entry["prob_all"] >= 0.5).astype(int)
    print("   (비교) 임계값 0.50 : Prec %.3f  Rec %.3f  F1 %.3f"
          % (precision_score(y_true, pred_half, zero_division=0),
             recall_score(y_true, pred_half), f1_score(y_true, pred_half)))
    print("✅ confusion matrix [[TN FP][FN TP]]:\n", confusion_matrix(y_true, pred))

# 동작 확인: 표본에서 이상거래 1건을 골라 판정해 본다
demo_entry = FIN_DATASETS["card"]
demo_id = int(next(i for i in demo_entry["pool"] if demo_entry["y"].iloc[i] == 1))
demo = predict_transaction("card", demo_id)

section("단일 건 판정 예시 (카드거래)")
print("✅ 행 번호 %d | 정답 %d | 예측 %d | 이상거래 확률 %.3f (임계값 %.2f)"
      % (demo["row_id"], demo["true"], demo["pred"], demo["prob"], demo["threshold"]))
print("✅ 판정 근거 top 6 (양수 = 이상 쪽으로 밀어낸 항목)")
print(pd.DataFrame(demo["contributions"]).round(3).to_string(index=False))
print("\n✅ intercept %.3f + 기여도 합 → logit %.3f → 확률 %.3f"
      % (demo["intercept"], demo["logit"], 1 / (1 + np.exp(-demo["logit"]))))
print("✅ ✔ 단일 건 판정 함수 준비 완료")

#### **9-4. 웹 화면 — 미학습 거래를 골라 이상거래인지 판정**

- 노트북 안에서 작은 **Flask 웹 서버**를 띄운다.
- 브라우저에서
  1. **전자금융공동망 / 카드거래** 탭을 고르고
  2. 목록에서 **거래 1건을 클릭**하면
  - 이상거래 확률과 판정(이상거래 의심 / 정상 거래)
  - 정답 라벨과 비교한 정오 판정, 이상거래 유형·설명
  - **판정 근거** (어떤 항목이 이 거래를 이상 쪽으로 밀었는가)
  - 거래 상세 필드 전체

  가 표시된다.
- **임계값 슬라이더**를 움직이면 holdout 전체에 대한 Precision·Recall·F1 이 즉시 다시 계산된다.
  낮추면 Recall↑ Precision↓, 높이면 그 반대가 되는 것을 눈으로 확인할 수 있다 → [실습 5]
- 목록에 뜨는 거래는 **모두 학습에 쓰이지 않은 건**이다.

In [ ]:
%pip install -q flask

import threading
import socket
import logging

from flask import Flask, request, jsonify, Response
from werkzeug.serving import make_server

logging.getLogger("werkzeug").setLevel(logging.ERROR)

PAGE = """<!doctype html>
<html lang="ko">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>금융 이상거래 판정 데모</title>
<style>
  :root { --line:#e3e6ea; --ink:#1f2933; --muted:#6b7684; --ok:#0f9d58; --bad:#d93025;
          --warn:#e8710a; --bg:#f6f7f9; --blue:#1a73e8; }
  * { box-sizing:border-box; }
  body { margin:0; padding:24px 16px 64px; background:var(--bg); color:var(--ink);
         font-family:"Malgun Gothic","Apple SD Gothic Neo","Noto Sans KR",sans-serif; }
  .wrap { max-width:1180px; margin:0 auto; }
  h1 { font-size:22px; margin:0 0 4px; }
  h2 { font-size:16px; margin:0 0 12px; }
  .sub { color:var(--muted); font-size:14px; }
  .muted { color:var(--muted); font-weight:400; }
  .card { background:#fff; border:1px solid var(--line); border-radius:12px; padding:20px; margin-bottom:18px; }
  .row { display:flex; gap:10px; align-items:center; flex-wrap:wrap; }
  .tabs { display:flex; gap:8px; margin-bottom:14px; }
  .tab { background:#fff; color:var(--ink); border:1px solid var(--line); border-radius:999px;
         padding:9px 18px; font-size:14px; cursor:pointer; }
  .tab.on { background:var(--blue); color:#fff; border-color:var(--blue); }
  button { background:var(--blue); color:#fff; border:0; border-radius:8px; padding:9px 16px;
           font-size:14px; cursor:pointer; }
  button.ghost { background:#fff; color:var(--ink); border:1px solid var(--line); }
  button:disabled { opacity:.5; cursor:default; }
  select { padding:8px 10px; border:1px solid var(--line); border-radius:8px; font-size:14px; background:#fff; }
  label.chk { font-size:13px; color:var(--muted); display:flex; align-items:center; gap:6px; cursor:pointer; }
  .srcnote { color:var(--muted); font-size:12px; margin-top:12px; line-height:1.7; }
  .tiles { display:flex; gap:12px; flex-wrap:wrap; margin-top:14px; }
  .tile { flex:1 1 130px; border:1px solid var(--line); border-radius:10px; padding:12px 14px; }
  .tile .k { font-size:12px; color:var(--muted); }
  .tile .v { font-size:22px; font-weight:700; margin-top:4px; }
  input[type=range] { flex:1 1 320px; }
  .split { display:grid; grid-template-columns:minmax(0,1.1fr) minmax(0,1fr); gap:16px; align-items:start; }
  @media (max-width:900px) { .split { grid-template-columns:1fr; } }
  .tablewrap { max-height:460px; overflow:auto; border:1px solid var(--line); border-radius:8px; }
  table { border-collapse:collapse; width:100%; font-size:13px; }
  th, td { border-bottom:1px solid var(--line); padding:7px 9px; text-align:center; white-space:nowrap; }
  th { background:#f2f4f7; font-weight:600; position:sticky; top:0; z-index:1; }
  tbody tr { cursor:pointer; }
  tbody tr:hover { background:#eef4ff; }
  tbody tr.on { background:#e8f0fe; box-shadow:inset 3px 0 0 var(--blue); }
  .hide { display:none; }
  .badge { display:inline-block; border-radius:999px; padding:3px 12px; font-size:12px; color:#fff; }
  .badge.ok { background:var(--ok); } .badge.bad { background:var(--bad); }
  .badge.na { background:#9aa0a6; } .badge.warn { background:var(--warn); }
  .verdict { border-radius:10px; padding:16px; margin-bottom:14px; }
  .verdict.alert { background:#fce8e6; border:1px solid #f3b8b2; }
  .verdict.safe { background:#e6f4ea; border:1px solid #a8d5b8; }
  .verdict .big { font-size:22px; font-weight:700; }
  .verdict .why { font-size:13px; color:var(--muted); margin-top:6px; }
  .bar { height:8px; background:#e8eaed; border-radius:4px; margin-top:8px; overflow:hidden; }
  .bar > span { display:block; height:100%; background:var(--blue); }
  .kv { font-size:13px; color:var(--muted); margin-top:6px; white-space:pre-line; }
  .contrib { width:100%; font-size:13px; border-collapse:collapse; }
  .contrib td { border:0; border-bottom:1px solid var(--line); padding:6px 8px; text-align:left; white-space:normal; }
  .contrib td.num { text-align:right; font-variant-numeric:tabular-nums; white-space:nowrap; }
  .cbar { height:6px; border-radius:3px; }
  .cbar.up { background:var(--bad); } .cbar.down { background:#5f6368; }
  .fields { display:grid; grid-template-columns:repeat(auto-fill,minmax(210px,1fr)); gap:6px 14px; font-size:13px; }
  .fields div { display:flex; justify-content:space-between; gap:10px; border-bottom:1px dotted var(--line); padding:4px 0; }
  .fields .fk { color:var(--muted); } .fields .fv { font-weight:600; }
  .note { color:var(--muted); font-size:12px; margin-top:10px; line-height:1.7; }
</style>
</head>
<body>
<div class="wrap">
  <h1>금융 이상거래 판정 데모</h1>
  <div class="sub">Logistic Regression(class_weight=&quot;balanced&quot;) &middot; 목록의 거래는 <b>모두 학습에 쓰이지 않은 건</b>이다.</div>

  <div class="card" style="margin-top:18px">
    <div class="tabs" id="tabs"></div>
    <div class="row">
      <select id="filter">
        <option value="all">전체 보기</option>
        <option value="fraud">이상거래(정답 1)만</option>
        <option value="normal">정상거래(정답 0)만</option>
      </select>
      <button id="shuffle" class="ghost">목록 새로 섞기</button>
      <button id="random">무작위 1건 판정</button>
      <label class="chk"><input type="checkbox" id="reveal"> 목록에 정답 라벨 표시</label>
      <span id="status" class="sub"></span>
    </div>
    <div class="srcnote" id="srcnote"></div>
  </div>

  <div class="card">
    <h2>판정 임계값 <span class="muted">— 확률이 이 값 이상이면 &quot;이상거래&quot;로 판정</span></h2>
    <div class="row">
      <input type="range" id="thr" min="0.01" max="0.99" step="0.01">
      <b id="thrval" style="min-width:52px">0.50</b>
      <button id="thrreset" class="ghost">권장 임계값으로</button>
    </div>
    <div class="tiles" id="scoretiles"></div>
    <div class="note" id="scorenote"></div>
  </div>

  <div class="split">
    <div class="card">
      <h2>미학습 거래 목록 <span class="muted">(행을 클릭하면 판정한다)</span></h2>
      <div class="tablewrap"><table id="list"></table></div>
    </div>
    <div class="card" id="result">
      <h2>판정 결과</h2>
      <div class="sub">왼쪽 목록에서 거래를 하나 고르세요.</div>
    </div>
  </div>
</div>

<script>
let INFO = null;
let kind = 'bank';
let rows = [];
let selected = null;
let last = null;
let seed = 42;

const $ = id => document.getElementById(id);
const pct = x => (100 * x).toFixed(1) + '%';
const esc = s => String(s).replace(/&/g, '&amp;').replace(/</g, '&lt;').replace(/>/g, '&gt;');

function ds() { return INFO.datasets[kind]; }

fetch('/api/info').then(r => r.json()).then(d => {
  INFO = d;
  $('tabs').innerHTML = Object.keys(d.datasets)
    .map(k => '<button class="tab" data-kind="' + k + '">' + esc(d.datasets[k].name) + '</button>').join('');
  document.querySelectorAll('.tab').forEach(b => b.addEventListener('click', () => switchKind(b.dataset.kind)));
  switchKind('bank');
});

function switchKind(k) {
  kind = k;
  selected = null;
  last = null;
  document.querySelectorAll('.tab').forEach(b => b.classList.toggle('on', b.dataset.kind === k));
  $('thr').value = ds().threshold;
  $('thrval').textContent = Number(ds().threshold).toFixed(2);
  $('srcnote').innerHTML = '미학습 데이터: <b>' + esc(ds().source) + '</b> &middot; 전체 ' +
    ds().n.toLocaleString() + '건 (이상 ' + ds().n_pos.toLocaleString() + '건, ' +
    (100 * ds().n_pos / ds().n).toFixed(2) + '%) &middot; 권장 임계값 ' + Number(ds().threshold).toFixed(2) +
    ' (Train 교차검증으로 결정) &middot; holdout PR-AUC ' + ds().pr_auc.toFixed(3) +
    ' / ROC-AUC ' + ds().roc_auc.toFixed(3);
  $('result').innerHTML = '<h2>판정 결과</h2><div class="sub">왼쪽 목록에서 거래를 하나 고르세요.</div>';
  loadRows();
  loadScore();
}

function loadRows() {
  const f = $('filter').value;
  $('status').textContent = '목록 불러오는 중...';
  fetch('/api/rows?kind=' + kind + '&filter=' + f + '&seed=' + seed + '&limit=60')
    .then(r => r.json())
    .then(d => {
      rows = d.rows;
      renderList(d.columns);
      $('status').textContent = d.rows.length + '건 표시';
    });
}

function renderList(columns) {
  const show = $('reveal').checked;
  let h = '<thead><tr><th>#</th>' + columns.map(c => '<th>' + esc(c) + '</th>').join('') +
          '<th class="ans' + (show ? '' : ' hide') + '">정답</th></tr></thead><tbody>';
  rows.forEach(r => {
    h += '<tr data-id="' + r.id + '">' +
         '<td class="muted">' + r.id + '</td>' +
         r.cells.map(c => '<td>' + esc(c) + '</td>').join('') +
         '<td class="ans' + (show ? '' : ' hide') + '">' +
         (r.true ? '<span class="badge bad">1</span>' : '<span class="badge na">0</span>') +
         '</td></tr>';
  });
  $('list').innerHTML = h + '</tbody>';
  $('list').querySelectorAll('tbody tr').forEach(tr => {
    tr.addEventListener('click', () => predict(Number(tr.dataset.id)));
    if (Number(tr.dataset.id) === selected) tr.classList.add('on');
  });
}

function predict(id) {
  selected = id;
  $('list').querySelectorAll('tbody tr').forEach(tr => tr.classList.toggle('on', Number(tr.dataset.id) === id));
  $('status').textContent = '판정 중...';
  fetch('/api/predict?kind=' + kind + '&id=' + id + '&threshold=' + $('thr').value)
    .then(r => r.json())
    .then(d => { last = d; renderResult(); $('status').textContent = '완료'; });
}

function renderResult() {
  if (!last) return;
  const thr = Number($('thr').value);
  const isAlert = last.prob >= thr;
  const correct = (isAlert ? 1 : 0) === last.true;

  let h = '<h2>판정 결과 <span class="muted">#' + last.row_id + '</span></h2>';
  h += '<div class="verdict ' + (isAlert ? 'alert' : 'safe') + '">' +
       '<div class="big">' + (isAlert ? '⚠ 이상거래 의심' : '✔ 정상 거래') + '</div>' +
       '<div class="why">이상거래 확률 <b>' + pct(last.prob) + '</b> ' +
       (isAlert ? '≥' : '&lt;') + ' 임계값 ' + thr.toFixed(2) + '</div>' +
       '<div class="bar"><span style="width:' + pct(Math.min(1, last.prob)) + ';background:' +
       (isAlert ? '#d93025' : '#0f9d58') + '"></span></div>' +
       '</div>';

  h += '<div class="row" style="gap:8px">' +
       (correct ? '<span class="badge ok">정답</span>' : '<span class="badge bad">오답</span>') +
       '<span class="sub">실제 라벨: <b>' + (last.true ? '이상거래(1)' : '정상거래(0)') + '</b></span></div>';
  if (last.true) {
    h += '<div class="kv">이상거래 유형: <b>' + esc(last.fraud_type) + '</b></div>';
    if (last.fraud_desc && last.fraud_desc !== '-') {
      h += '<div class="kv">설명: ' + esc(last.fraud_desc) + '</div>';
    }
  }

  const maxc = Math.max.apply(null, last.contributions.map(c => Math.abs(c.c))) || 1;
  h += '<h2 style="margin-top:22px">판정 근거 <span class="muted">— logit 기여도 상위 6개</span></h2><table class="contrib">';
  last.contributions.forEach(c => {
    const up = c.c > 0;
    h += '<tr><td>' + esc(c.name) +
         '<div class="cbar ' + (up ? 'up' : 'down') + '" style="width:' +
         (100 * Math.abs(c.c) / maxc).toFixed(1) + '%"></div></td>' +
         '<td class="num muted">z ' + c.z.toFixed(2) + ' × w ' + c.w.toFixed(2) + '</td>' +
         '<td class="num" style="color:' + (up ? '#d93025' : '#5f6368') + '"><b>' +
         (up ? '+' : '') + c.c.toFixed(3) + '</b></td></tr>';
  });
  h += '</table>';
  h += '<div class="note">절편 ' + last.intercept.toFixed(3) + ' + 모든 기여도의 합 = logit ' +
       last.logit.toFixed(3) + ' → sigmoid → 확률 ' + pct(last.prob) +
       '. <span style="color:#d93025">붉은 막대</span>는 이 거래를 <b>이상</b> 쪽으로, ' +
       '<span style="color:#5f6368">회색 막대</span>는 <b>정상</b> 쪽으로 밀어낸 항목이다.</div>';

  h += '<h2 style="margin-top:22px">거래 상세</h2><div class="fields">';
  last.fields.forEach(f => {
    h += '<div><span class="fk">' + esc(f.k) + '</span><span class="fv">' + esc(f.v) + '</span></div>';
  });
  h += '</div>';

  $('result').innerHTML = h;
}

function loadScore() {
  const thr = $('thr').value;
  fetch('/api/score?kind=' + kind + '&threshold=' + thr)
    .then(r => r.json())
    .then(d => {
      $('scoretiles').innerHTML =
        tile('Precision', d.precision.toFixed(3)) +
        tile('Recall', d.recall.toFixed(3)) +
        tile('F1', d.f1.toFixed(3)) +
        tile('이상 판정 건수', d.n_flagged.toLocaleString() + '건') +
        tile('놓친 이상거래(FN)', d.fn.toLocaleString() + '건');
      $('scorenote').innerHTML =
        '미학습 ' + d.n.toLocaleString() + '건 전체를 임계값 ' + Number(thr).toFixed(2) +
        '으로 채점한 결과 &middot; TN ' + d.tn.toLocaleString() + ' / FP ' + d.fp.toLocaleString() +
        ' / FN ' + d.fn.toLocaleString() + ' / TP ' + d.tp.toLocaleString() +
        '<br>임계값을 낮추면 놓치는 이상거래(FN)는 줄지만 헛짚는 정상거래(FP)가 늘어난다. 비용이 큰 쪽을 기준으로 고른다.';
    });
}

let scoreTimer = null;
$('thr').addEventListener('input', () => {
  $('thrval').textContent = Number($('thr').value).toFixed(2);
  renderResult();
  clearTimeout(scoreTimer);
  scoreTimer = setTimeout(loadScore, 180);
});
$('thrreset').addEventListener('click', () => {
  $('thr').value = ds().threshold;
  $('thrval').textContent = Number(ds().threshold).toFixed(2);
  renderResult();
  loadScore();
});
$('filter').addEventListener('change', loadRows);
$('reveal').addEventListener('change', () => {
  document.querySelectorAll('.ans').forEach(td => td.classList.toggle('hide', !$('reveal').checked));
});
$('shuffle').addEventListener('click', () => { seed = Math.floor(Math.random() * 100000); loadRows(); });
$('random').addEventListener('click', () => {
  if (!rows.length) return;
  predict(rows[Math.floor(Math.random() * rows.length)].id);
});

function tile(k, v) {
  return '<div class="tile"><div class="k">' + k + '</div><div class="v">' + v + '</div></div>';
}
</script>
</body>
</html>
"""

app = Flask(__name__)


@app.route("/")
def index():
    return Response(PAGE, mimetype="text/html")


@app.route("/api/info")
def api_info():
    datasets = {}
    for key, entry in FIN_DATASETS.items():
        m = entry["metrics"]
        datasets[key] = {
            "name": entry["name"],
            "source": entry["source"],
            "threshold": entry["threshold"],
            "n": m["n"],
            "n_pos": m["n_pos"],
            "pr_auc": m["pr_auc"],
            "roc_auc": m["roc_auc"],
        }
    return jsonify({"datasets": datasets})


@app.route("/api/rows")
def api_rows():
    entry = FIN_DATASETS[request.args.get("kind", "bank")]
    mode = request.args.get("filter", "all")
    limit = max(1, min(300, int(request.args.get("limit", 60))))
    labels = entry["y"].to_numpy()

    ids = [int(i) for i in entry["pool"]]
    if mode == "fraud":
        ids = [i for i in ids if labels[i] == 1]
    elif mode == "normal":
        ids = [i for i in ids if labels[i] == 0]

    rng = np.random.RandomState(int(request.args.get("seed", 42)))
    rng.shuffle(ids)
    ids = ids[:limit]

    raw = entry["raw"]
    rows = [
        {
            "id": i,
            "true": int(labels[i]),
            "cells": [fmt_value(raw.iloc[i][c]) for c in entry["list_cols"]],
        }
        for i in ids
    ]
    return jsonify({"columns": entry["list_cols"], "rows": rows})


@app.route("/api/predict")
def api_predict():
    return jsonify(predict_transaction(
        request.args.get("kind", "bank"),
        int(request.args.get("id", 0)),
        threshold=float(request.args.get("threshold", 0.5)),
    ))


@app.route("/api/score")
def api_score():
    entry = FIN_DATASETS[request.args.get("kind", "bank")]
    thr = float(request.args.get("threshold", 0.5))
    y_true = entry["y"].to_numpy()
    pred = (entry["prob_all"] >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()

    return jsonify({
        "n": int(len(y_true)),
        "threshold": thr,
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "n_flagged": int(pred.sum()),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    })


def find_free_port(preferred=7870):
    for port in range(preferred, preferred + 20):
        with socket.socket() as probe:
            if probe.connect_ex(("127.0.0.1", port)) != 0:
                return port
    return 0


# 이 셀을 여러 번 실행해도 서버가 중복으로 뜨지 않도록 이전 서버를 정리한다.
if globals().get("FIN_WEB_SERVER") is not None:
    FIN_WEB_SERVER.shutdown()
    FIN_WEB_SERVER.server_close()

FIN_WEB_PORT = find_free_port()
FIN_WEB_SERVER = make_server("127.0.0.1", FIN_WEB_PORT, app)
FIN_WEB_THREAD = threading.Thread(target=FIN_WEB_SERVER.serve_forever, daemon=True)
FIN_WEB_THREAD.start()

FIN_WEB_URL = f"http://127.0.0.1:{FIN_WEB_PORT}/"

print("✅ 웹 데모 주소 :", FIN_WEB_URL)
for entry in FIN_DATASETS.values():
    print("   %s : 미학습 %d건 중 %d건을 목록에 표시 (권장 임계값 %.2f)"
          % (entry["name"], len(entry["y"]), len(entry["pool"]), entry["threshold"]))
print("   서버를 멈추려면 9-5 셀을 실행한다. (커널을 종료해도 함께 멈춘다)")

try:
    import webbrowser
    webbrowser.open(FIN_WEB_URL)
except Exception as exc:
    print("브라우저 자동 실행 실패. 위 주소를 직접 열어라:", exc)

#### **9-5. 웹 서버 종료**

실습이 끝나면 포트를 반납한다. (노트북 커널을 종료해도 함께 멈춘다)

In [ ]:
if globals().get("FIN_WEB_SERVER") is not None:
    FIN_WEB_SERVER.shutdown()
    FIN_WEB_SERVER.server_close()
    FIN_WEB_SERVER = None
    print("✅ ✔ 웹 서버를 종료했습니다.")
else:
    print("실행 중인 웹 서버가 없습니다.")